# To Gauge Mitigation Failure Cases But Looking For Consecutive Failures

In [6]:
import pandas as pd
import numpy as np
import os, glob

def check_consecutive_pair_0(df):
    # Checks for consecutive pairs of 0 in df (this algo only works for checking pairs of 0, can't work for higher n consecutive 0s)
    if "Control_State" in df.columns:
        col_name = "Control_State"
    else:
        col_name = "Unnamed: 2"
    df_0 = df.loc[(df[col_name] == "Waypoint") & (df["Measured_Throughput"] == 0)]
    diff_time_0 = df_0["Measurement_Time"].diff()
    return np.any(diff_time_0 == 1)

DATASET_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Obj3_Retransmission_Datasets/Fixed_Prob_MANET_Interference/manet_scenario_2"
SWITCH_TIME = "/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/switch_time/fp_manet_scenario_2_switch_time.csv"
# DATASET_PATH = "/home/rlim0005/FANET_Dataset/DJISpark_Obj3_Retransmission_Datasets/DV_Based_500_MANET_Interference/manet_scenario_0"
# SWITCH_TIME = "/home/rlim0005/FANET_Dataset/DJISpark_Obj3_Retransmission_Datasets/switch_time/dv_500_manet_scenario_0_switch_time.csv"

switch_time_df = pd.read_csv(SWITCH_TIME)
scenario_list = switch_time_df["Scenario"].values
run_num_list = switch_time_df["Run_Num"].values
total_dl_fail = 0
total_gw_fail = 0
total_ul_fail = 0
for i in range(len(scenario_list)):
    dl_fail = 0
    # gw_fail = 0
    uav_csv_files = glob.glob(os.path.join(DATASET_PATH, scenario_list[i], "Run-{}_UAV-*-Throughput.csv".format(run_num_list[i])))
    for file in uav_csv_files:
        df = pd.read_csv(file)
        if check_consecutive_pair_0(df):
            dl_fail = 1
    # gw_csv_file = os.path.join(DATASET_PATH, scenario_list[i], "Run-{}_GW-Throughput.csv".format(run_num_list[i]))
    # gw_df = pd.read_csv(gw_csv_file)
    # if check_consecutive_pair_0(gw_df):
    #     dl_fail = 1
    #     gw_fail = 1
    ul_fail = 0
    ul_csv_file = os.path.join(DATASET_PATH, scenario_list[i], "Run-{}_GCS-Throughput.csv".format(run_num_list[i]))
    ul_df = pd.read_csv(ul_csv_file)
    if check_consecutive_pair_0(ul_df):
        ul_fail = 1
    
    total_dl_fail += dl_fail
    total_gw_fail += dl_fail
    total_ul_fail += ul_fail

print("Percentage DL Success: {}".format(100 - total_dl_fail/len(scenario_list) * 100))
# print("Percentage GW Success: {}".format(100 - total_gw_fail/len(scenario_list) * 100))
print("Percentage UL Success: {}".format(100 - total_ul_fail/len(scenario_list) * 100))

Percentage DL Success: 42.03921568627451
Percentage UL Success: 100.0
